# ResLSTM-SER: Speech Emotion Recognition using Attention-based LSTM-Network with Residual Connection

Reproducible training notebook for the DSPA 2026 paper:
**"Speech Emotion Recognition using Attention-based LSTM-Network with Residual Connection"**
by Daniil Krasnoproshin and Maxim Vashkevich.

> IEEE Xplore: https://ieeexplore.ieee.org/document/11476771

This notebook performs:
1. Feature extraction (MFCC + chromagram) from RAVDESS .wav files
2. Global mean/std normalization
3. Model training with Optuna hyperparameter optimization
4. 10-run statistical evaluation with optimal hyperparameters
5. K-fold cross-validation (speaker-independent, 5-fold)

## Imports & Path Configuration

In [ ]:
import glob
import os
import sys
from typing import List, Tuple

import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from ignite.metrics.recall import Recall
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from tqdm import tqdm

# ── Paths — ADJUST THESE ──
# The RAVDESS dataset is NOT included in this repository.
# Download from: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
# or: https://zenodo.org/record/1188976

ROOT_DIR = os.path.abspath('')  # current working directory
RAVDESS_DATA_PATH = os.path.join(ROOT_DIR, 'data', 'ravdess', 'audio_speech_actors_01-24', '**', '*.wav')
EXTRACTED_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'ravdess', 'extracted')
MODEL_BACKUP_DIR = os.path.join(ROOT_DIR, 'model_backup')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"RAVDESS path: {RAVDESS_DATA_PATH}")

## Reproducibility

In [ ]:
def set_seed(seed: int = 42):
    """Set random seed for reproducibility."""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## RAVDESS Label Mappings

In [ ]:
def get_emotions():
    return ["neutral", "calm", "happy", "sad", "angry", "fear", "disgust", "surprised"]

def get_emotions_dictionary():
    return {
        "01": "neutral",
        "02": "calm",
        "03": "happy",
        "04": "sad",
        "05": "angry",
        "06": "fear",
        "07": "disgust",
        "08": "surprised",
    }

def get_actors():
    return {f"{i:02d}": str(i) for i in range(1, 25)}

## Feature Extraction (wav → npy)

In [ ]:
def extract_features(file_path: str,
                     n_mfcc: int = 34,
                     n_chroma: int = 12,
                     n_fft: int = 4096,
                     hop_length: int = None) -> np.ndarray:
    """
    Extract MFCC and chromagram features from an audio file.

    Args:
        file_path: Path to the audio file.
        n_mfcc: Number of MFCC features to extract.
        n_chroma: Number of chroma bins to produce.
        n_fft: FFT window size.
        hop_length: Number of samples between successive frames
                    (default: n_fft // 2).

    Returns:
        features: (time_len, n_mfcc + n_chroma) feature array, or None on error.
    """
    import librosa

    try:
        audio, sample_rate = librosa.load(file_path, sr=None)

        if hop_length is None:
            hop_length = n_fft // 2

        mfcc = librosa.feature.mfcc(
            y=audio, sr=sample_rate, n_mfcc=n_mfcc,
            n_fft=n_fft, hop_length=hop_length
        ).T

        chroma = librosa.feature.chroma_stft(
            y=audio, sr=sample_rate, n_fft=n_fft,
            n_chroma=n_chroma, hop_length=hop_length
        ).T

        features = np.concatenate([mfcc, chroma], axis=1)
        return features
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

### Two-pass Feature Extraction with Global Normalization

In [ ]:
def process_dataset(dataset_path: str,
                    x_output_path_template: str,
                    y_output_path_template: str,
                    id_output_path_template: str,
                    n_mfcc: int,
                    n_chroma: int,
                    n_fft: int,
                    hop_length: int):
    """
    Two-pass feature extraction with global mean/std normalization.

    Pass 1: accumulate global statistics (mean, std) across all files.
    Pass 2: normalize each file and save as .npy.
    """
    valid_files = []
    valid_emotions = []
    valid_actors = []

    emotions_dict = get_emotions_dictionary()
    actors_dict = get_actors()
    observed_emotions = get_emotions()

    print("Pass 1: collecting global statistics...")
    total_sum = None
    total_sq_sum = None
    total_samples = 0

    all_files = list(glob.glob(dataset_path))
    if not all_files:
        raise FileNotFoundError(
            f"No audio files found at {dataset_path}. "
            f"Please download the RAVDESS dataset first."
        )

    for file in tqdm(all_files, desc="Computing statistics"):
        file_name = os.path.basename(file)
        splitted = file_name.split("-")
        emotion = emotions_dict[splitted[2]]

        if emotion not in observed_emotions:
            continue

        actor = actors_dict[splitted[6].split(".")[0]]
        features = extract_features(file_path=file, n_mfcc=n_mfcc,
                                    n_chroma=n_chroma, n_fft=n_fft,
                                    hop_length=hop_length)

        if features is None:
            continue

        valid_files.append(file)
        valid_emotions.append(emotion)
        valid_actors.append(actor)

        n_samples = features.shape[0]
        if total_sum is None:
            n_features = features.shape[1]
            total_sum = np.zeros(n_features, dtype=np.float64)
            total_sq_sum = np.zeros(n_features, dtype=np.float64)

        total_sum += np.sum(features, axis=0)
        total_sq_sum += np.sum(features ** 2, axis=0)
        total_samples += n_samples

    if total_sum is None or total_samples == 0:
        raise RuntimeError("No valid data found.")

    global_mean = total_sum / total_samples
    global_var = (total_sq_sum / total_samples) - (global_mean ** 2)
    global_std = np.sqrt(np.maximum(global_var, 1e-8))

    print(f"Global stats: {total_samples} frames, {len(global_mean)} features")
    print(f"Mean (first 5): {global_mean[:5]}")
    print(f"Std  (first 5): {global_std[:5]}")

    os.makedirs(EXTRACTED_DATA_DIR, exist_ok=True)
    np.save(os.path.join(EXTRACTED_DATA_DIR, "global_mean.npy"), global_mean)
    np.save(os.path.join(EXTRACTED_DATA_DIR, "global_std.npy"), global_std)

    print("Pass 2: normalizing and saving...")
    for i, (file, emotion, actor) in enumerate(tqdm(
            zip(valid_files, valid_emotions, valid_actors),
            desc="Normalizing",
            total=len(valid_files)
    )):
        features = extract_features(file_path=file, n_mfcc=n_mfcc,
                                    n_chroma=n_chroma, n_fft=n_fft,
                                    hop_length=hop_length)
        features_norm = (features - global_mean) / global_std

        x_path = x_output_path_template.format(i, n_mfcc, n_chroma, n_fft, hop_length)
        np.save(x_path, features_norm)

    y_path = y_output_path_template.format(n_mfcc, n_chroma, n_fft, hop_length)
    np.save(y_path, np.array(valid_emotions))

    id_path = id_output_path_template.format(n_mfcc, n_chroma, n_fft)
    np.save(id_path, np.array(valid_actors))

    print(f"Processed {len(valid_files)} files.")
    return global_mean, global_std

### Run Feature Extraction

In [ ]:
N_MFCC = 34
N_CHROMA = 12
N_FFT = 4096
HOP_LENGTH = N_FFT // 2

os.makedirs(EXTRACTED_DATA_DIR, exist_ok=True)

x_template = os.path.join(EXTRACTED_DATA_DIR, "X_{}_mfcc_{}_n_chroma_{}_nfft_{}_hop_size_{}.npy")
y_template = os.path.join(EXTRACTED_DATA_DIR, "y_labels_mfcc_{}_n_chroma_{}_nfft_{}_hop_size_{}.npy")
id_template = os.path.join(EXTRACTED_DATA_DIR, "IDs_mfcc_{}_n_chroma_{}_nfft_{}.npy")

y_path = y_template.format(N_MFCC, N_CHROMA, N_FFT, HOP_LENGTH)

if not os.path.exists(y_path):
    process_dataset(
        dataset_path=RAVDESS_DATA_PATH,
        x_output_path_template=x_template,
        y_output_path_template=y_template,
        id_output_path_template=id_template,
        n_mfcc=N_MFCC,
        n_chroma=N_CHROMA,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )
else:
    print(f"Features already extracted at {EXTRACTED_DATA_DIR}")

## PyTorch Dataset

In [ ]:
class RavdessAudioDataset(Dataset):
    """RAVDESS dataset for variable-length sequences with speaker-based folds."""

    def __init__(self,
                 mfccs_dir: str,
                 annotations_file: str,
                 actor_ids_file: str,
                 desired_labels: list = None):
        self.mfccs_dir = mfccs_dir
        self.label_mapping = {
            'neutral': 0, 'calm': 1, 'happy': 2, 'sad': 3,
            'angry': 4, 'fear': 5, 'disgust': 6, 'surprised': 7
        }

        y = np.load(annotations_file, allow_pickle=True)
        ids = np.load(actor_ids_file, allow_pickle=True)

        if desired_labels is not None:
            mask = np.isin(y, desired_labels)
            y = y[mask]
            ids = ids[mask]

        self.y = torch.tensor([self.label_mapping[label] for label in y], dtype=torch.long)
        self.X_ids = [int(a) for a in ids]

        self.feature_files = []
        for i in range(len(self.y)):
            fpath = mfccs_dir.format(i, N_MFCC, N_CHROMA, N_FFT, HOP_LENGTH)
            self.feature_files.append(fpath)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        features = np.load(self.feature_files[index])
        features = torch.tensor(features, dtype=torch.float32)
        label = self.y[index]
        return features, label

    def get_kth_fold_inds(self, fold_num: int):
        """
        Speaker-independent 5-fold split (same as paper).
        Returns (train_indices, val_indices, test_indices).
        """
        folds = [
            [2, 5, 14, 15, 16],
            [3, 6, 7, 13, 18],
            [10, 11, 12, 19, 20],
            [8, 17, 21, 23, 24],
            [1, 4, 9, 22],
        ]
        folds_val = [
            [3, 6],
            [10, 11],
            [8, 17],
            [1, 14],
            [2, 5],
        ]

        ids_train, ids_val, ids_test = [], [], []
        for i in range(len(self.X_ids)):
            if self.X_ids[i] in folds[fold_num]:
                ids_test.append(i)
            elif self.X_ids[i] in folds_val[fold_num]:
                ids_val.append(i)
            else:
                ids_train.append(i)
        return ids_train, ids_val, ids_test


def pad_mfcc_sequence(batch: List[Tuple[torch.Tensor, int]]):
    """Collate function: pad sequences and return (padded, labels, lengths)."""
    sequences, labels = zip(*batch)
    lengths = torch.tensor([seq.size(0) for seq in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True)
    labels = torch.stack(labels) if isinstance(labels[0], torch.Tensor) else torch.tensor(labels, dtype=torch.long)
    return padded, labels, lengths

### Create Dataset Instance

In [ ]:
feature_dir_template = os.path.join(EXTRACTED_DATA_DIR, "X_{}_mfcc_{}_n_chroma_{}_nfft_{}_hop_size_{}.npy")
id_path = id_template.format(N_MFCC, N_CHROMA, N_FFT)

dataset = RavdessAudioDataset(
    mfccs_dir=feature_dir_template,
    annotations_file=y_path,
    actor_ids_file=id_path,
    desired_labels=get_emotions()
)
print(f"Dataset: {len(dataset)} samples, 8 classes")

## Models

### ResLSTM-SA (Residual LSTM with Soft Attention)

In [ ]:
class ResLSTM_SA(nn.Module):
    """
    ResLSTM-SA: LSTM with residual connection and soft attention.

    Architecture (from the paper):
        Input → LSTM1 ──(+)──→ LSTM2 → Attention → FC → Output
    """

    def __init__(self,
                 input_size: int,
                 hidden_size: int,
                 num_layers: int,
                 num_classes: int,
                 dropout_p: float = 0.1,
                 device: str = 'cpu'):
        super().__init__()
        self.device = device
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.num_layers = num_layers

        # LSTM1: input_size → input_size (for residual connection)
        self.lstm1 = nn.LSTM(input_size, input_size, num_layers,
                             bidirectional=False, batch_first=True)
        # LSTM2: input_size → hidden_size
        self.lstm2 = nn.LSTM(input_size, hidden_size, num_layers,
                             bidirectional=False, batch_first=True)
        # Single-vector soft attention
        self.attention_vector = nn.Parameter(torch.empty(hidden_size, 1))
        # BatchNorm
        self.bn_residual = nn.BatchNorm1d(input_size)
        self.bn = nn.BatchNorm1d(hidden_size)
        # Classifier
        self.fc = nn.Linear(hidden_size, num_classes)
        self.drop = nn.Dropout(p=dropout_p)
        self.num_classes = num_classes
        self.initialize_weights()

    def initialize_weights(self):
        for lstm, proj_size in [(self.lstm1, self.input_size),
                                (self.lstm2, self.hidden_size)]:
            for layer in range(self.num_layers):
                nn.init.xavier_normal_(getattr(lstm, f'weight_ih_l{layer}'))
                nn.init.orthogonal_(getattr(lstm, f'weight_hh_l{layer}'))
                bias_ih = getattr(lstm, f'bias_ih_l{layer}')
                bias_hh = getattr(lstm, f'bias_hh_l{layer}')
                nn.init.zeros_(bias_ih)
                nn.init.zeros_(bias_hh)
                # Forget gate bias = 1
                with torch.no_grad():
                    bias_ih[proj_size:2 * proj_size].fill_(1.0)
                    bias_hh[proj_size:2 * proj_size].fill_(1.0)
        nn.init.xavier_normal_(self.attention_vector)
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def compute_attention(self, lstm_out):
        """Soft attention over time dimension."""
        scores = torch.matmul(lstm_out, self.attention_vector).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(lstm_out * weights.unsqueeze(-1), dim=1)
        return context, weights

    def forward(self, x, lengths, return_attention=False):
        batch_size = x.size(0)
        x_original = x.clone()

        # LSTM1
        h0 = torch.zeros(self.num_layers, batch_size, self.input_size, device=self.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.input_size, device=self.device)
        x_packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                        enforce_sorted=False)
        lstm1_out_packed, _ = self.lstm1(x_packed, (h0, c0))
        lstm1_out, _ = pad_packed_sequence(lstm1_out_packed, batch_first=True)

        if lstm1_out.size(1) != x.size(1):
            seq_len = min(lstm1_out.size(1), x.size(1))
            lstm1_out = lstm1_out[:, :seq_len, :]
            x_original = x_original[:, :seq_len, :]

        # Residual + BatchNorm
        residual = lstm1_out + x_original
        residual_norm = self.bn_residual(residual.transpose(1, 2)).transpose(1, 2)

        # LSTM2
        h0_l2 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=self.device)
        c0_l2 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=self.device)
        residual_packed = pack_padded_sequence(residual_norm, lengths.cpu(),
                                               batch_first=True, enforce_sorted=False)
        lstm2_out_packed, _ = self.lstm2(residual_packed, (h0_l2, c0_l2))
        lstm2_out, _ = pad_packed_sequence(lstm2_out_packed, batch_first=True)

        # Attention + classification
        context, attention_weights = self.compute_attention(lstm2_out)
        context = self.bn(context)
        logits = self.fc(self.drop(context))

        if return_attention:
            return logits, attention_weights
        return logits

### LSTM-SA Baseline (single LSTM, no residual connection)

In [ ]:
class LSTM_SA(nn.Module):
    """Baseline LSTM-SA from Mirsamadi & Barsoum (2017)."""

    def __init__(self,
                 input_size: int,
                 hidden_size: int,
                 num_layers: int,
                 num_classes: int,
                 dropout_p: float = 0.1,
                 device: str = 'cpu'):
        super().__init__()
        self.device = device
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            bidirectional=False, batch_first=True)
        self.attention_vector = nn.Parameter(torch.empty(hidden_size, 1))
        self.bn = nn.BatchNorm1d(hidden_size)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.drop = nn.Dropout(p=dropout_p)
        self.num_classes = num_classes
        self.initialize_weights()

    def initialize_weights(self):
        for layer in range(self.num_layers):
            nn.init.xavier_normal_(getattr(self.lstm, f'weight_ih_l{layer}'))
            nn.init.orthogonal_(getattr(self.lstm, f'weight_hh_l{layer}'))
            bias_ih = getattr(self.lstm, f'bias_ih_l{layer}')
            bias_hh = getattr(self.lstm, f'bias_hh_l{layer}')
            nn.init.zeros_(bias_ih)
            nn.init.zeros_(bias_hh)
            with torch.no_grad():
                bias_ih[self.hidden_size:2 * self.hidden_size].fill_(1.0)
                bias_hh[self.hidden_size:2 * self.hidden_size].fill_(1.0)
        nn.init.xavier_normal_(self.attention_vector)
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x, lengths):
        batch_size = x.size(0)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=self.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=self.device)
        x_packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                        enforce_sorted=False)
        lstm_out_packed, _ = self.lstm(x_packed, (h0, c0))
        lstm_out, _ = pad_packed_sequence(lstm_out_packed, batch_first=True)

        scores = torch.matmul(lstm_out, self.attention_vector).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(lstm_out * weights.unsqueeze(-1), dim=1)
        context = self.bn(context)
        logits = self.fc(self.drop(context))
        return logits

## Evaluation Utilities

In [ ]:
from ignite.engine import Engine

def _eval_step(engine, batch):
    return batch

def get_default_evaluator():
    return Engine(_eval_step)

def get_test_evaluator():
    return Engine(_eval_step)

## Training Loop

In [ ]:
def training_loop(model, loss_fn, optimizer, lr_scheduler,
                  train_loader, val_loader, n_epochs,
                  model_path):
    """Single training loop with validation."""
    best_epoch = -1
    UAR_val_max = -1
    loss_train_history = np.zeros(n_epochs)
    loss_val_history = np.zeros(n_epochs)

    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

    default_evaluator = get_default_evaluator()
    macro_metric = Recall(average=True)
    macro_metric.attach(default_evaluator, "macro_recall")

    for epoch in range(n_epochs):
        model.train()
        loss_train = 0.0
        for X, labels, lengths in train_loader:
            X, labels, lengths = X.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
            optimizer.zero_grad()

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    logits = model(X, lengths)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(X, lengths)
                loss = loss_fn(logits, labels)
                loss.backward()
                optimizer.step()
            loss_train += loss.item()
        loss_train /= len(train_loader)

        # Validation
        model.eval()
        loss_val = 0.0
        val_pred_all, val_true_all = [], []
        with torch.no_grad():
            for X, labels, lengths in val_loader:
                X, labels, lengths = X.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
                logits = model(X, lengths)
                loss = loss_fn(logits, labels)
                loss_val += loss.item()
                val_pred_all.append(torch.softmax(logits, dim=1))
                val_true_all.append(labels)

        val_pred_all = torch.cat(val_pred_all)
        val_true_all = torch.cat(val_true_all)
        loss_val /= len(val_loader)
        loss_train_history[epoch] = loss_train
        loss_val_history[epoch] = loss_val

        default_evaluator.terminate()
        state = default_evaluator.run([[val_pred_all, val_true_all]])
        UAR_val = state.metrics["macro_recall"]

        if lr_scheduler is not None:
            lr_scheduler.step()

        if UAR_val > UAR_val_max:
            UAR_val_max = UAR_val
            best_epoch = epoch
            torch.save(model.state_dict(), model_path)

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{n_epochs} | "
                  f"Train loss: {loss_train:.4f} | "
                  f"Val loss: {loss_val:.4f} | "
                  f"Val UAR: {UAR_val:.4f}")

    model.load_state_dict(torch.load(model_path))
    return loss_train_history, loss_val_history, best_epoch, UAR_val_max

## K-Fold Cross-Validation

In [ ]:
def k_fold_cv(dataset, model_factory, loss_fn, optimizer_factory,
              lr_scheduler_factory, n_epochs, k_fold, batch_size,
              init_model_path, model_path):
    """K-fold cross-validation returning out-of-fold predictions."""
    oof_preds, oof_targets = [], []

    for fold in range(k_fold):
        print(f"\n{'='*60}")
        print(f"Fold {fold + 1}/{k_fold}")
        print(f"{'='*60}")

        inds_train, inds_val, inds_test = dataset.get_kth_fold_inds(fold)
        train_set = torch.utils.data.Subset(dataset, inds_train)
        val_set = torch.utils.data.Subset(dataset, inds_val)
        test_set = torch.utils.data.Subset(dataset, inds_test)

        train_loader = torch.utils.data.DataLoader(
            train_set, batch_size=batch_size, shuffle=True, collate_fn=pad_mfcc_sequence)
        val_loader = torch.utils.data.DataLoader(
            val_set, batch_size=batch_size, shuffle=False, collate_fn=pad_mfcc_sequence)
        test_loader = torch.utils.data.DataLoader(
            test_set, batch_size=batch_size, shuffle=False, collate_fn=pad_mfcc_sequence)

        set_seed(42 + fold * 100)
        model = model_factory()
        torch.save(model.state_dict(), init_model_path)
        optimizer = optimizer_factory(model)
        lr_scheduler = lr_scheduler_factory(optimizer)

        training_loop(model, loss_fn, optimizer, lr_scheduler,
                      train_loader, val_loader, n_epochs, model_path)

        model.eval()
        fold_preds, fold_targets = [], []
        with torch.no_grad():
            for X, labels, lengths in test_loader:
                X, labels, lengths = X.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
                logits = model(X, lengths)
                fold_preds.append(torch.softmax(logits, dim=1).cpu())
                fold_targets.append(labels.cpu())
        oof_preds.append(torch.cat(fold_preds))
        oof_targets.append(torch.cat(fold_targets))

    return oof_preds, oof_targets

## Hyperparameter Optimization with Optuna

### Configure Search Space

In [ ]:
INPUT_SIZE = N_MFCC + N_CHROMA  # 46
NUM_CLASSES = 8
NUM_LAYERS = 1
N_EPOCHS = 100
K_FOLD = 5

# Choose hidden size: 32, 64, or 128
HIDDEN_SIZE = 64

SQLITE_STORAGE_URL = f"sqlite:///optuna_h{HIDDEN_SIZE}.db"
STUDY_NAME = f"ResLSTM_SA_H{HIDDEN_SIZE}_tuning"
print(f"Hidden size: {HIDDEN_SIZE}, Study: {STUDY_NAME}")

### Objective Function

In [ ]:
def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-5, 3e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16, 32, 64])
    dropout_rate = trial.suggest_float("dropout_rate", 0.02, 0.5)
    weight_decay = trial.suggest_float("wd", 2e-5, 4e-2, log=True)
    t_0_loop = trial.suggest_categorical("t_0_loop", [1, 2, 3, 5, 10])

    set_seed(42)

    def model_factory():
        return ResLSTM_SA(
            input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, num_classes=NUM_CLASSES,
            dropout_p=dropout_rate, device=str(DEVICE)
        ).to(DEVICE)

    def optimizer_factory(model):
        return optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    def scheduler_factory(optimizer):
        return optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=t_0_loop, T_mult=1, eta_min=lr * 0.01)

    loss_fn = nn.CrossEntropyLoss()
    os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)
    init_model_path = os.path.join(MODEL_BACKUP_DIR, f"init_{STUDY_NAME}.pt")
    model_path = os.path.join(MODEL_BACKUP_DIR, f"best_{STUDY_NAME}.pt")

    oof_preds, oof_targets = k_fold_cv(
        dataset=dataset, model_factory=model_factory, loss_fn=loss_fn,
        optimizer_factory=optimizer_factory, lr_scheduler_factory=scheduler_factory,
        n_epochs=N_EPOCHS, k_fold=K_FOLD, batch_size=batch_size,
        init_model_path=init_model_path, model_path=model_path)

    all_preds = torch.cat([p for p in oof_preds])
    all_targets = torch.cat([t for t in oof_targets])

    evaluator = get_default_evaluator()
    macro_metric = Recall(average=True)
    macro_metric.attach(evaluator, "macro_recall")
    evaluator.terminate()
    state = evaluator.run([[all_preds, all_targets]])
    return state.metrics["macro_recall"]

### Run Optuna Study

In [ ]:
N_TRIALS = 100  # Number of Optuna trials

study = optuna.create_study(
    storage=SQLITE_STORAGE_URL,
    study_name=STUDY_NAME,
    direction="maximize",
    load_if_exists=True,
)
print(f"Starting Optuna optimization with {N_TRIALS} trials...")
study.optimize(objective, n_trials=N_TRIALS)

print("\nBest hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"  Best UAR: {study.best_value:.4f}")

## 10-Run Statistical Evaluation

Run 10 independent training cycles with the best hyperparameters to get mean ± std UAR.

In [ ]:
os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

# Load best params from Optuna
study = optuna.load_study(study_name=STUDY_NAME, storage=SQLITE_STORAGE_URL)
best_params = study.best_params
print("Using hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

results = []
for run in range(10):
    print(f"\n{'='*60}")
    print(f"Run {run + 1}/10")
    print(f"{'='*60}")

    set_seed(42 + run * 100)

    def model_factory():
        return ResLSTM_SA(
            input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, num_classes=NUM_CLASSES,
            dropout_p=best_params['dropout_rate'], device=str(DEVICE)
        ).to(DEVICE)

    def optimizer_factory(model):
        return optim.Adam(model.parameters(), lr=best_params['lr'],
                          weight_decay=best_params['wd'])

    def scheduler_factory(optimizer):
        return optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=best_params['t_0_loop'], T_mult=1,
            eta_min=best_params['lr'] * 0.01)

    loss_fn = nn.CrossEntropyLoss()
    model_name = f"ResLSTM_SA_H{HIDDEN_SIZE}"
    init_model_path = os.path.join(MODEL_BACKUP_DIR, f"init_{model_name}_run{run}.pt")
    model_path = os.path.join(MODEL_BACKUP_DIR, f"best_{model_name}_run{run}.pt")

    oof_preds, oof_targets = k_fold_cv(
        dataset=dataset, model_factory=model_factory, loss_fn=loss_fn,
        optimizer_factory=optimizer_factory, lr_scheduler_factory=scheduler_factory,
        n_epochs=N_EPOCHS, k_fold=K_FOLD, batch_size=best_params['batch_size'],
        init_model_path=init_model_path, model_path=model_path)

    all_preds = torch.cat([p for p in oof_preds])
    all_targets = torch.cat([t for t in oof_targets])

    evaluator = get_default_evaluator()
    macro_metric = Recall(average=True)
    macro_metric.attach(evaluator, "macro_recall")
    evaluator.terminate()
    state = evaluator.run([[all_preds, all_targets]])
    uar = state.metrics["macro_recall"]

    _m = model_factory()
    num_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)

    results.append({"run": run + 1, "uar": uar, "num_params": num_params})
    print(f"Run {run + 1}: UAR = {uar:.4f}, Params = {num_params}")

# Statistics
uars = [r["uar"] for r in results]
mean_uar = np.mean(uars)
std_uar = np.std(uars)
print(f"\nFinal: UAR = {mean_uar:.4f} +/- {std_uar:.4f}")
print(f"Num params: {results[0]["num_params"]}")

## Summary

This notebook reproduces the full experimental pipeline from the DSPA 2026 paper:

1. **Feature Extraction**: 34 MFCC + 12 chroma = 46-dim features, two-pass global normalization
2. **Model**: ResLSTM-SA — residual LSTM with soft attention
3. **Hyperparameter Tuning**: Bayesian optimization with Optuna (TPE sampler)
4. **Evaluation**: 10 independent runs, speaker-independent 5-fold CV, reporting UAR mean ± std

### Key Results (from paper)

| Model | Hidden Size | Params | UAR (mean ± std) | Max UAR |
|-------|-------------|--------|------------------|---------|
| ResLSTM-SA-h64 | 64 | 46.8k | 0.6232 ± 0.0119 | **0.6517** |
| ResLSTM-SA-h128 | 128 | 108.9k | 0.6107 ± 0.0134 | 0.6348 |
| ResLSTM-SA-h32 | 32 | 28.0k | 0.6130 ± 0.0111 | 0.6315 |

### References

- Krasnoproshin, D. & Vashkevich, M. (2026). *Speech Emotion Recognition using Attention-based LSTM-Network with Residual Connection.* DSPA 2026.
- Mirsamadi, S., Barsoum, E., & Zhang, C. (2017). *Automatic speech emotion recognition using recurrent neural networks with local attention.* ICASSP 2017.
- Livingstone, S. R. & Russo, F. A. (2018). *The RAVDESS database.* PLoS ONE.
- Akiba, T. et al. (2019). *Optuna: A Next-generation Hyperparameter Optimization Framework.* KDD 2019.